# A1 — Baseline zero-shot (không train)

Chấm Qwen3-VL-8B **chưa fine-tune** trên tập Vi-VQA, rồi đọc tay 30 mẫu.

Mục đích: trả lời câu **"fine-tune có đáng không"** trước khi đốt 8–20 giờ GPU.
Toàn bộ notebook chạy trong ~30–45 phút, không train gì cả.

**Cần:** Runtime GPU + HF token đã được cấp quyền vào dataset.

## 1. GPU

T4 (Colab free) **không hỗ trợ bfloat16** — cell dưới sẽ tự chọn dtype phù hợp.

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), 'Bật GPU: Runtime → Change runtime type → GPU'
gpu = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
supports_bf16 = torch.cuda.is_bf16_supported()

print(f'{gpu.name} — {vram:.1f} GB — bf16: {supports_bf16}')

# T4 (Turing) không có bf16; Ampere trở lên thì có.
DTYPE = 'bfloat16' if supports_bf16 else 'float16'

# Trọng số 8B ở 16-bit chiếm ~16GB. Phần VRAM còn lại phải chứa vision
# token của ảnh — ảnh càng nét càng nhiều token.
if vram >= 22:
    MAX_PIXELS = 1310720   # 1280 token — độ phân giải đầy đủ
elif vram >= 15:
    MAX_PIXELS = 589824    # 576 token
else:
    MAX_PIXELS = 262144    # 256 token

print(f'→ dtype={DTYPE}  image_max_pixels={MAX_PIXELS}')
if vram < 15:
    print('⚠️  VRAM thấp cho model 8B ở 16-bit — nếu OOM thì phải load 4-bit')

## 2. Cài đặt

In [ ]:
!git clone https://github.com/theAbyssOfTime2004/Vi-VQA.git 2>/dev/null || echo 'đã clone'
%cd Vi-VQA
!git checkout claude/project-overview-2qeyep

# Chỉ cài phần cần cho inference. Extra [train] kéo theo deepspeed và
# bitsandbytes — phải build, mất thêm nhiều phút, và A1 không train gì.
# transformers>=4.57 bắt buộc: Qwen3VLForConditionalGeneration không có ở bản thấp hơn.
!pip install -q -e '.[data,infer]'

# Không cài flash-attn: build trên Colab rất lâu, và code tự lùi về SDPA.
# Với 200 mẫu thì chênh lệch tốc độ không đáng công chờ.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

!huggingface-cli whoami

## 3. Chuẩn bị dữ liệu (streaming)

`data.streaming=true` + `--limit` chỉ kéo về đúng số record cần, thay vì
tải nguyên split vài GB. Với 400 record ta được ~1.200 cặp QA — thừa cho A1.

In [ ]:
!vivqa prepare --limit 400 --set data.streaming=true

!du -sh data/images 2>/dev/null

## 4. Chấm base model

`--model-path` nhận thẳng HF model id, không cần checkpoint.
`temperature=0` (mặc định) → greedy, chạy lại ra kết quả y hệt.
Grounding **tắt** — đây là baseline zero-shot trung thực.

Dùng split `train` vì với `--limit 400` thì val chỉ còn ~20 ảnh. Model này
chưa train nên không có chuyện rò rỉ — mọi mẫu đều là zero-shot với nó.

### ⏱️ Về thời gian

Chi phí chi phối là **số token sinh ra**, không phải kích thước ảnh. Model
gốc chưa fine-tune nói rất dài — có thể 300–400 token cho một câu mà đáp
án chuẩn chỉ ~15–20 token. Với `max_new_tokens=512` thì ra ~15 giây/mẫu
trên L4, tức ~50 phút cho 200 mẫu.

Cắt `max_new_tokens` xuống 128 nhanh gấp ~3 lần và **không làm hỏng phép
đo**: đáp án chuẩn ngắn hơn thế nhiều, nên phần bị cắt là phần thừa. Câu
trả lời bị cắt cụt lại chính là tín hiệu cho ô 1 — model nói quá dài.

Muốn giữ nguyên độ dài đầy đủ (để đo mức độ dài dòng chính xác) thì bỏ
dòng `max_new_tokens` đi và chấp nhận chờ lâu hơn.

In [ ]:
!vivqa eval \
    --model-path Qwen/Qwen3-VL-8B-Instruct \
    --split train \
    --num-samples 200 \
    --output ./results/baseline.json \
    --set model.torch_dtype={DTYPE} \
    --set model.image_max_pixels={MAX_PIXELS} \
    --set inference.max_new_tokens=128

## 5. ⚠️ Đọc con số — nhưng đừng tin nó

Exact match gần như chắc chắn rất thấp **kể cả khi model trả lời đúng hết**,
vì đáp án tham chiếu do Gemini viết có văn phong rất đặc trưng.

Con số ở đây chỉ là mốc để so sánh sau này. Dữ liệu thật nằm ở mục 6.

In [ ]:
import json

result = json.load(open('results/baseline.json'))
print(f"Đã chấm {result['num_samples']} mẫu (lỗi: {result['num_failed']})\n")
for name, value in result['metrics'].items():
    suffix = '' if name == 'cider' else '%'
    print(f'  {name:<12} {value:8.2f}{suffix}')

### Độ dài trả lời — bằng chứng định lượng cho ô 1

Nếu model gốc trả lời dài gấp nhiều lần đáp án chuẩn, thì phần lớn thứ
fine-tune sẽ dạy nó là **nói ngắn lại theo kiểu Gemini** — tức là văn phong,
không phải năng lực. Đây là cách đo điều đó mà không cần đọc mẫu nào.

In [ ]:
import statistics

pred_len = [len(p['prediction']) for p in result['predictions']]
ref_len = [len(p['reference']) for p in result['predictions']]

mp, mr = statistics.median(pred_len), statistics.median(ref_len)
print(f'Độ dài trung vị — dự đoán: {mp:.0f} ký tự | đáp án chuẩn: {mr:.0f} ký tự')
print(f'Tỉ lệ dài dòng: {mp / mr:.1f}x\n')

# Bao nhiêu câu bị chặn bởi max_new_tokens? Cụt = model định nói còn dài hơn.
truncated = sum(1 for p in result['predictions'] if not p['prediction'].rstrip().endswith(('.', '!', '?')))
print(f'Câu không kết thúc bằng dấu câu (nhiều khả năng bị cắt): {truncated}/{len(pred_len)}')

if mp / mr > 3:
    print('\n→ Model gốc dài dòng hơn hẳn. Ủng hộ mạnh giả thuyết ô 1 (văn phong).')
elif mp / mr > 1.5:
    print('\n→ Dài hơn vừa phải. Đọc 30 mẫu ở mục 6 để phân định.')
else:
    print('\n→ Độ dài đã tương đương. Chênh lệch điểm số (nếu có) không đến từ độ dài.')


## 6. ★ Đọc tay 30 mẫu — phần quan trọng nhất

Phân mỗi mẫu vào **đúng một** ô:

| Ô | Dấu hiệu | Nghĩa |
|---|----------|-------|
| **1** | Đúng nội dung, khác cách nói | Fine-tune chỉ mua **văn phong** |
| **2** | Đọc đúng chữ nhưng trả lời sai | Lỗi **suy luận** |
| **3** | Đọc sai chữ trên biển / sai dấu / bịa tên | **OCR yếu** — fine-tune sửa được thật |

Ghi số đếm vào cell cuối.

In [ ]:
for i, p in enumerate(result['predictions'][:30], 1):
    print(f"{'─' * 70}\n[{i}]  {p['id']}")
    print(f"  Q   {p['question']}")
    print(f"  GT  {p['reference']}")
    print(f"  PR  {p['prediction']}")
print('─' * 70)

### Xem kèm ảnh (cho những mẫu nghi ngờ OCR)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show(i):
    """Hiện ảnh + câu hỏi + đáp án của mẫu thứ i (1-based)."""
    p = result['predictions'][i - 1]
    plt.figure(figsize=(8, 8))
    plt.imshow(Image.open(f"data/images/{p['image']}")); plt.axis('off'); plt.show()
    print(f"Q   {p['question']}\nGT  {p['reference']}\nPR  {p['prediction']}")

show(1)

## 7. Kết luận

Điền số đếm rồi chạy — cell sẽ nói bạn nên làm gì tiếp.

In [ ]:
o1 = 0   # đúng nội dung, khác cách nói
o2 = 0   # đọc đúng chữ, suy luận sai
o3 = 0   # đọc sai chữ trên ảnh

total = o1 + o2 + o3
assert total > 0, 'Điền số đếm trước đã'

print(f'Ô1 văn phong : {o1:2d}  ({o1/total:5.1%})')
print(f'Ô2 suy luận  : {o2:2d}  ({o2/total:5.1%})')
print(f'Ô3 OCR       : {o3:2d}  ({o3/total:5.1%})\n')

if o1 / total > 0.6:
    print('→ Chênh lệch before/after chủ yếu là VĂN PHONG.')
    print('  Bỏ full train. Chỉ cần LoRA nhỏ (~1.500 mẫu, 1 epoch) để minh hoạ,')
    print('  rồi chuyển hướng bài thành "benchmark này đo phong cách hay năng lực?".')
elif o3 / total > 0.2:
    print('→ OCR tiếng Việt yếu thật — có dư địa cải thiện NĂNG LỰC.')
    print('  Full fine-tune có lý. Nên chấm riêng nhóm câu hỏi scene-text để đo.')
else:
    print('→ Lỗi chủ yếu là suy luận. Kiểm tra chất lượng nhãn Gemini trước khi train:')
    print('  nhãn sinh từ description, không có người verify, có thể chứa nhiễu.')